# Analysis of Caloric Density and its Predicted Online Rating

**Name(s)**: Eric Zhao, Dayoung In

**Website Link**: https://eericzzhao.github.io/Analysis-of-Caloric-Density-and-its-Predicted-Online-Rating/

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

import plotly.express as px
pd.options.plotting.backend = 'plotly'

# from dsc80_utils import * # Feel free to uncomment and use this.

## Step 1: Introduction

We chose this one because as students, we often have to cook food ourselves. Given the steady rise of prices in restaurants and "eating out" becomes more and more expensive everyday, more and more households are deciding to cook instead. This has become a challenge as recipes are often not standardized. Thus, the usage of reviews is very much necessary in order to determine whether a recipe was worth doing or not. This lead us to wonder what in the recipe of a food item or more specifically it's calories, sugar, or fat determines the rating that it gets. This serves as the basis as to why we chose this dataset. 

This is important as this shows that there is obvious a clear preference for specific recipes that contain different levels of nutritional factors. This is helpful for chefs and people making these recipes to better adjust their recipes so that more people are willing to try it, such as if their original recipe was just missing or containing too much of a certain ingredient. 

After merging, we can see that there were 83782 rows and 19 columns. The relevant columns are average rating (this is what we're trying to predict), calories, sugar_pdv, total_fat_pdv (and saturated_fat_pdv). These are what we're testing for in terms of what is a bigger factor in affecting a recipe's score. 

## Step 2: Data Cleaning and Exploratory Data Analysis

In [ ]:
recipes = pd.read_csv('RAW_recipes.csv')
interactions = pd.read_csv('interactions.csv') 

# Left merge the recipes with the interactions/ratings (gives us all the reviews for every recipe)
merged_df = recipes.merge(interactions, left_on='id', right_on='recipe_id', how='left')

# Fill ratings of 0 with np.nan (we don't want NaN values to affect our calculations nor is it useful)
merged_df['rating'] = merged_df['rating'].replace(0, np.nan)

# We take the average rating (value) for every recipe (index)
avg_ratings = merged_df.groupby('id')['rating'].mean().rename('avg_rating')

# 5. Create a new column for the average rating (left merge helps us avoid recipes with no ratings)
recipes = recipes.merge(avg_ratings, left_on='id', right_index=True, how='left')

# 6. Check the nutrition column (so we can split it into multiple columns
nut_cols = [
    'calories', 'total_fat_pdv', 'sugar_pdv', 'sodium_pdv', 
    'protein_pdv', 'saturated_fat_pdv', 'carbohydrates_pdv'
]
nutrition_data = recipes['nutrition'].str.strip('[]').str.split(', ', expand=True).astype(float)
nutrition_data.columns = nut_cols
recipes = pd.concat([recipes.drop('nutrition', axis=1), nutrition_data], axis=1)

print(f"Final Shape: {recipes.shape}")
print(f"Sample of Cleaned Data:\n{recipes[['name', 'avg_rating', 'calories']].head()}")

# Save to CSV 
recipes.to_csv('cleaned_recipes.csv', index=False)

In [ ]:
### Univariate Analysis
import os

# Distribution of Average Ratings 
fig_ratings = px.histogram(
    recipes, 
    x='avg_rating', 
    nbins=20,
    title='Distribution of Average Recipe Ratings',
    labels={'avg_rating': 'Average Star Rating'},
    template='plotly_white'
)
# Save as HTML so we can embed it
fig_ratings.write_html('assets/ratings_dist.html', include_plotlyjs='cdn')

fig_ratings.show()

**Interpretation:** The distribution of average ratings is significantly left-skewed, with a massive concentration of recipes receiving a perfect 5.0 score. This suggests a "positivity bias" common in online recipe platforms, where users are more likely to rate recipes they enjoyed.

In [ ]:
# Distribution of Calories 

# Clipped at 2000 beacuse the calories are heavily skewed
fig_calories = px.histogram(
    recipes[recipes['calories'] < 2000], 
    x='calories', 
    nbins=50,
    title='Distribution of Calories (Clipped at 2000)',
    labels={'calories': 'Calories per Serving'},
    template='plotly_white',
    color_discrete_sequence=['indianred']
)
fig_calories.write_html('assets/calories_dist.html', include_plotlyjs='cdn')
fig_calories.show()

**Interpretation:** Calories follow a right-skewed distribution. While the majority of recipes fall between 200 and 600 calories (typical for a standard meal), there is a "long tail" of high-calorie recipes that likely represent large-batch items or high-density desserts.

In [ ]:
### Bivariate Analysis

# 1. Calorie Category (Above vs Below Median) 
median_calories = recipes['calories'].median()
recipes['calorie_density'] = recipes['calories'].apply(
    lambda x: 'Above Median' if x > median_calories else 'Below Median'
)

# Average Rating by Calorie Density (Box Plot) 
# This helps us identify if 'high calorie' foods actually get better ratings.
fig_bivariate_box = px.box(
    recipes.dropna(subset=['avg_rating']), 
    x='calorie_density', 
    y='avg_rating',
    title='Average Rating: Above-Median vs. Below-Median Calories',
    labels={'calorie_density': 'Caloric Density', 'avg_rating': 'Average Rating'},
    template='plotly_white',
    color='calorie_density',
    color_discrete_map={'Above Median': 'firebrick', 'Below Median': 'seagreen'}
)
fig_bivariate_box.write_html('assets/rating_vs_calories_box.html', include_plotlyjs='cdn')
fig_bivariate_box.show()

In [ ]:
# Plot 2: Calories vs. Sugar PDV (Scatter Plot) 
# We are checking if if 'unhealthy' food traits are correlated. (Outliers were removed for better visualization)
filtered_df = recipes[(recipes['calories'] < 2000) & (recipes['sugar_pdv'] < 200)]
fig_bivariate_scatter = px.scatter(
    filtered_df,
    x='sugar_pdv',
    y='calories',
    opacity=0.4,
    title='Relationship Between Sugar Content and Total Calories',
    labels={'sugar_pdv': 'Sugar (% Daily Value)', 'calories': 'Total Calories'},
    template='plotly_white',
    trendline="ols"
)
fig_bivariate_scatter.write_html('assets/sugar_vs_calories_scatter.html', include_plotlyjs='cdn')
fig_bivariate_scatter.show()

## Step 3: Assessment of Missingness

In [ ]:
# Create a column to check if True if we have a missing rating 
merged_df['rating_is_missing'] = merged_df['rating'].isna()

def ks_permutation_test(data, col_to_check, missing_col, n_permutations=500):
    """
    We do a permutation test using the KS statistic to check if the distribution
    of col_to_check is different when missing_col is True or False.
    """
    from scipy.stats import ks_2samp
    
    # Observed Statistic
    gr1 = data.loc[data[missing_col], col_to_check]
    gr2 = data.loc[~data[missing_col], col_to_check]
    obs_stat = ks_2samp(gr1, gr2).statistic
    
    # Permutations
    stats = []
    shuffled = data.copy()
    for _ in range(n_permutations):
        shuffled[missing_col] = np.random.permutation(shuffled[missing_col])
        s1 = shuffled.loc[shuffled[missing_col], col_to_check]
        s2 = shuffled.loc[~shuffled[missing_col], col_to_check]
        stats.append(ks_2samp(s1, s2).statistic)
        
    p_val = np.mean(np.array(stats) >= obs_stat)
    return obs_stat, p_val

# Test 1: Doesrating missingness depend on the 'minutes' column?
clean_minutes = merged_df[merged_df['minutes'] < 1000]
obs_min, p_min = ks_permutation_test(clean_minutes, 'minutes', 'rating_is_missing')

# Test 2: Does rating missingness depend on the 'n_steps' column?
obs_steps, p_steps = ks_permutation_test(merged_df, 'n_steps', 'rating_is_missing')

print(f"Minutes Dependency: p-value = {p_min}")
print(f"N_Steps Dependency: p-value = {p_steps}")

# Generate a plot for the 'Does Depend' case
fig_missing = px.histogram(
    clean_minutes, 
    x='minutes', 
    color='rating_is_missing', 
    histnorm='probability density', 
    title='Distribution of Minutes: Rating Missing vs. Not Missing',
    labels={'minutes': 'Minutes', 'rating_is_missing': 'Is Rating Missing?'},
    template='plotly_white', 
    barmode='overlay'
)
fig_missing.write_html('assets/missingness_dist.html', include_plotlyjs='cdn')
fig_missing.show()

## Step 4: Hypothesis Testing

In [ ]:
# We only care about recipes that actually have an average rating
hyp_data = recipes[['calorie_density', 'avg_rating']].dropna()

# Calculate Observed Statistic: (Mean of Above Median) - (Mean of Below Median)
means = hyp_data.groupby('calorie_density')['avg_rating'].mean()
obs_diff = means['Above Median'] - means['Below Median']

# Run our Permutation Test
n_permutations = 1000
shuffled_diffs = []

for _ in range(n_permutations):
    # shuffle the calorie labels
    shuffled_labels = np.random.permutation(hyp_data['calorie_density'])
    
    # Calculate difference in means for the shuffled data
    temp_df = hyp_data.assign(shuffled=shuffled_labels)
    shuffled_means = temp_df.groupby('shuffled')['avg_rating'].mean()
    diff = shuffled_means['Above Median'] - shuffled_means['Below Median']
    shuffled_diffs.append(diff)

# NOW we can calculate our P-value (Two-sided test)
p_value = np.mean(np.abs(shuffled_diffs) >= np.abs(obs_diff))
print(f"Observed Difference: {obs_diff}")
print(f"P-value: {p_value}")

# Visualization for our Empirical Distribution
fig_hyp = px.histogram(
    pd.DataFrame({'diffs': shuffled_diffs}), 
    x='diffs', 
    title='Empirical Distribution of Mean Differences',
    labels={'diffs': 'Difference in Mean Rating (Above - Below)'},
    template='plotly_white'
)
fig_hyp.add_vline(x=obs_diff, line_dash="dash", line_color="red", annotation_text="Observed Difference")
fig_hyp.write_html('assets/hypothesis_test_dist.html', include_plotlyjs='cdn')
fig_hyp.show()

## Step 5: Framing a Prediction Problem

In [ ]:
# Define the binary target variable (Let's predict if a recipe is 5 stars or not)
recipes['is_perfect_rating'] = (recipes['avg_rating'] == 5.0).astype(int)

# Define X features and y target 
# Baseline features: one numeric (calories) and one complexity (n_steps)
X = recipes[['calories', 'n_steps', 'total_fat_pdv', 'sugar_pdv']]
y = recipes['is_perfect_rating']

# Classic train/est Split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# Check for class imbalance
print(y_train.value_counts(normalize=True))

## Step 6: Baseline Model

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.impute import SimpleImputer

# 1. Update X to include ALL raw columns needed for engineering
features_for_final = ['calories', 'n_steps', 'sugar_pdv', 'minutes']
X = recipes[features_for_final]
y = recipes['is_perfect_rating']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# Refined Feature Engineering function
def engineer_features(df):
    df = df.copy()
    # We add 1 to denominators to avoid division by zero
    df['sugar_ratio'] = df['sugar_pdv'] / (df['calories'] + 1)
    df['steps_per_minute'] = df['n_steps'] / (df['minutes'] + 1)
    # Return the specific columns the ColumnTransformer expects
    return df[['calories', 'n_steps', 'sugar_ratio', 'steps_per_minute']]

# Pipeline
preprocessor = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]), ['calories', 'n_steps', 'sugar_ratio', 'steps_per_minute'])
])

final_pipe = Pipeline([
    ('engineer', FunctionTransformer(engineer_features)),
    ('preproc', preprocessor),
    ('rf', RandomForestClassifier(random_state=42))
])

# Hyperparameter Tuning (Grid Search)
param_grid = {
    'rf__n_estimators': [50, 100],
    'rf__max_depth': [5, 10], 
    'rf__min_samples_split': [2, 5]
}

# 5. Run the Search
grid_search = GridSearchCV(final_pipe, param_grid, cv=3, scoring='f1', n_jobs=-1)
grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_
print(f"Best Parameters: {grid_search.best_params_}")
print(f"Best Training F1-Score: {grid_search.best_score_:.4f}")

## Step 7: Final Model

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import f1_score, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# Grab the rest of the raw features for the final model
raw_features = ['calories', 'n_steps', 'sugar_pdv', 'total_fat_pdv', 'minutes']
X = recipes[raw_features]
y = recipes['is_perfect_rating']

# Ensure we use the same split as the baseline for a fair comparison
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# Define the Feature Engineering Function
def engineer_nutritional_ratios(df):
    df = df.copy()
    # Feature 1: Sugar relative to calories (is it a 'sugar bomb'?)
    df['sugar_ratio'] = df['sugar_pdv'] / (df['calories'] + 1)
    # Feature 2: Fat relative to calories (is it 'greasy'?)
    df['fat_ratio'] = df['total_fat_pdv'] / (df['calories'] + 1)
    return df[['calories', 'n_steps', 'sugar_ratio', 'fat_ratio']]

# Build the Pipeline (we can now classify)
preprocessor = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]), ['calories', 'n_steps', 'sugar_ratio', 'fat_ratio'])
])

final_pipe = Pipeline([
    ('engineer', FunctionTransformer(engineer_nutritional_ratios)),
    ('preproc', preprocessor),
    ('rf', RandomForestClassifier(random_state=42))
])

# Hyperparameter Tuning (depth to control overfitting)
param_grid = {
    'rf__max_depth': [5, 10, 15],
    'rf__n_estimators': [50, 100],
    'rf__min_samples_split': [2, 5]
}

grid_search = GridSearchCV(final_pipe, param_grid, cv=3, scoring='f1', n_jobs=-1)
grid_search.fit(X_train, y_train)

# Final Evaluation
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

print(f"Best Hyperparameters: {grid_search.best_params_}")
print(f"Final Test F1-Score: {f1_score(y_test, y_pred):.4f}")

# Visualization
ConfusionMatrixDisplay.from_estimator(best_model, X_test, y_test, cmap='Reds')
plt.title("Confusion Matrix for Final Model")
plt.show()

## Step 8: Fairness Analysis

In [ ]:
from sklearn.metrics import precision_score

# Prepare data (test set and the calorie_density labels) for fairness analysis
results = X_test.copy()
results['prediction'] = best_model.predict(X_test)
results['tag'] = y_test
# Re-create the calorie_density binary label for the test set
median_cal = recipes['calories'].median()
results['is_high_calorie'] = (results['calories'] > median_cal).astype(int)

# Define the metric function
def get_precision(df):
    return precision_score(df['tag'], df['prediction'], zero_division=0)

# Observed Statistic
obs_high = get_precision(results[results['is_high_calorie'] == 1])
obs_low = get_precision(results[results['is_high_calorie'] == 0])
obs_diff = abs(obs_high - obs_low)

# Permutation Test
n_perms = 1000
diffs = []

for _ in range(n_perms):
    # We shuffle the 'is_high_calorie' labels
    shuffled_labels = np.random.permutation(results['is_high_calorie'])
    temp_results = results.assign(shuffled=shuffled_labels)
    
    # Compute precision for shuffled groups
    p_high = get_precision(temp_results[temp_results['shuffled'] == 1])
    p_low = get_precision(temp_results[temp_results['shuffled'] == 0])
    diffs.append(abs(p_high - p_low))

# Calculate P-value
p_val = np.mean(np.array(diffs) >= obs_diff)

print(f"Observed Difference in Precision: {obs_diff:.4f}")
print(f"P-value: {p_val:.4f}")

# Visualization
fig_fair = px.histogram(
    pd.DataFrame({'diffs': diffs}), x='diffs', 
    title='Fairness Analysis: Permutation Test for Precision Parity',
    labels={'diffs': 'Absolute Difference in Precision (High - Low)'},
    template='plotly_white'
)
fig_fair.add_vline(x=obs_diff, line_dash="dash", line_color="red")
fig_fair.write_html('assets/fairness_test.html', include_plotlyjs='cdn')